# Лабораторная работа 5. Очистка и нормализация больших объемов текстовых данных

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/l05.ipynb)

**Цели занятия:**

- Очищать «грязные» тексты из веба и соцсетей: HTML-теги и сущности, ссылки, хештеги, эмодзи, служебные фразы
- Нормализовать регистр, пробелы и апострофы, удалять пустые строки, точные и нечёткие дубликаты
- Транслитерировать узбекский текст с кириллицы на латиницу по официальным правилам
- Обрабатывать 100 000 строк эффективно: векторные методы `.str`, обработка уникальных значений, чтение по частям

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Откуда берётся «грязь»

Тексты для Big Data собирают с сайтов, из Telegram-каналов и соцсетей. В них много того, что мешает анализу. Файл `data/news_raw.csv` (столбцы id, source, date, raw_text) содержит типичные проблемы:

| Проблема | Пример из данных | Решение |
|---|---|---|
| HTML-теги и сущности | `<p>`, `<b>YANGILIK:</b>`, `&nbsp;` | `html.unescape` + регулярка или BeautifulSoup |
| ссылки | `Batafsil: https://t.me/...` | `https?://\S+` и `www\.\S+` |
| хештеги, эмодзи | `#yangilik`, 👍 🔥 | `#\w+`, диапазоны Unicode эмодзи |
| служебные фразы | `(manba: www.talaba-news.uz)` | список шаблонов для источника |
| регистр, пробелы | `SUN'IY  INTELLEKT`, табуляция | `.lower()`, `\s+` → пробел |
| апострофы | o'zbek / o‘zbek / oʻzbek | нормализация к ʻ и ʼ |
| пустые строки, дубликаты | `""`, `"   "`, повторы, перепосты | фильтр и `drop_duplicates` |
| другая письменность | Ўзбек тили… | транслитерация |

## Порядок шагов важен

```python
def clean_text(text):
    if not isinstance(text, str):          # NaN -> ""
        return ""
    text = html.unescape(text)              # 1. &nbsp; &amp; -> символы
    for rx in (TAG, BOILER, URL, HASHTAG, EMOJI):
        text = rx.sub(" ", text)            # 2. мусор -> ПРОБЕЛ (не пустая строка!)
    text = normalize(text).lower()          # 3. апострофы, регистр
    return re.sub(r"\s+", " ", text).strip()  # 4. пробелы — в самом конце
```

- Служебные фразы удаляются **до** ссылок: шаблон `(manba: ...)` содержит адрес, и если сначала удалить ссылку, останется обрывок «(manba: ».
- Мусор заменяется **пробелом**, иначе слова склеятся: `<p>Salom</p><p>Dunyo` → «SalomDunyo».
- Пробелы схлопываются **последними**: после удаления тегов и эмодзи остаются лишние пробелы. Шаблон `\s` в Python понимает и неразрывный пробел `\xa0`, который получается из `&nbsp;`.

### Регулярные выражения или BeautifulSoup?

`BeautifulSoup(text, "html.parser").get_text(" ")` корректно разбирает любой HTML, включая вложенные и «сломанные» теги, комментарии, скрипты. Регулярка `<[^>]+>` намного быстрее, но ошибается на сложной разметке (например, `>` внутри атрибута). Для коротких фрагментов из соцсетей хватает регулярки; для целых веб-страниц используйте парсер.

## Дубликаты: точные и нечёткие

```python
df = df[df["clean"] != ""]                        # пустые после очистки
df = df.drop_duplicates(subset="raw_text")        # точные копии
df = df.drop_duplicates(subset="clean")           # совпадают после нормализации
```

**Нечёткие дубликаты** (перепосты с другим оформлением) находятся только после нормализации: «<p>BU YANGILIK 🔥» и «bu yangilik #news» дают одинаковый `clean`. На миллиардах документов сравнивают не строки, а **хеши** (`hashlib.md5(clean)`), а похожие, но не одинаковые тексты ищут методами **MinHash / LSH**.

## Транслитерация кириллица → латиница

Официальный алфавит 1995 года (с правками): большинство букв переводится один к одному, особые случаи:

| Кириллица | Латиница | Кириллица | Латиница |
|---|---|---|---|
| ў | oʻ | ш | sh |
| ғ | gʻ | ч | ch |
| қ | q | ё / ю / я | yo / yu / ya |
| ҳ | h | х | x |
| ъ | ʼ | ь | (не пишется) |
| **е** | **ye** в начале слова и после гласной, иначе **e** | **ц** | **ts** после гласной, иначе **s** |

*Ер → Yer, поезд → poyezd, келди → keldi; Транслитерация → Transliteratsiya, цирк → sirk.* Заглавная буква даёт заглавную первую букву сочетания: *Ш → Sh, Ў → Oʻ*.

## Масштаб: 100 000 строк и больше

| Способ | Идея | Скорость |
|---|---|---|
| цикл `for i, row in df.iterrows()` | построчно, с созданием объекта Series на каждую строку | очень медленно |
| `s.apply(f)` / `s.map(f)` | функция Python для каждого значения | приемлемо |
| `s.str.replace(...)`, `s.str.lower()` | «векторные» строковые методы pandas | так же или немного быстрее, короче код |
| **`unique` + `map(dict)`** | обработать только уникальные тексты и разнести результат | в десятки раз быстрее при повторах |

> **Заметка: Строки в pandas — не numpy**  
> Для чисел векторные операции выполняются в C и ускоряют код в сотни раз. Строки (тип object) pandas всё равно обрабатывает циклом Python, поэтому `.str` даёт удобство, а не чудо-скорость. Настоящее ускорение — не делать лишнюю работу: в потоках соцсетей много повторов (репосты, шаблонные сообщения), и чистить каждый уникальный текст достаточно **один раз**.

Если данные не помещаются в память, читайте их **по частям** и обрабатывайте генератором:

```python
def clean_chunks(path, chunksize=20_000):
    for chunk in pd.read_csv(path, chunksize=chunksize):   # не весь файл сразу
        chunk["clean"] = chunk["raw_text"].map(clean_text)
        yield chunk                                         # отдаём часть и «засыпаем»

for i, chunk in enumerate(clean_chunks("big.csv")):
    chunk.to_csv("clean.csv", mode="a", header=(i == 0), index=False)
```

Так же устроены Spark и Dask: данные делятся на **партиции**, каждая обрабатывается независимо, а результат собирается в конце.

## Задание 1. Осмотр грязных данных

Прочитайте `data/news_raw.csv` в `raw` и оцените масштаб проблем в столбце `raw_text`. Посчитайте число строк:

- `n_rows` — всего;
- `n_empty` — пустых: NaN или только пробельные символы;
- `n_html` — с HTML-тегом (шаблон `<[^>]+>`);
- `n_urls` — со ссылкой (`https?://` или `www.`);
- `n_emoji` — с эмодзи (шаблон `EMOJI_RE` в стартовом коде);
- `n_cyr` — с кириллицей (шаблон `CYR_RE`);
- `n_dup` — полных дубликатов текста (`duplicated(subset="raw_text")`).

Для поиска используйте `raw["raw_text"].str.contains(шаблон, na=False)`.

<details><summary>💡 Подсказка</summary>

Пустые: `text.fillna("").str.strip().eq("").sum()`. Ссылки: `text.str.contains(r"https?://|www\.", na=False).sum()`. `na=False` нужен, чтобы NaN считались как «нет совпадения».

</details>

In [ ]:
import pandas as pd

EMOJI_RE = "[\U0001F300-\U0001FAFF\u2600-\u27BF]"     # диапазоны эмодзи
CYR_RE = "[а-яёўқғҳА-ЯЁЎҚҒҲ]"                          # кириллица с узбекскими буквами

raw = ...
text = raw["raw_text"]

n_rows = ...
n_empty = ...
n_html = ...
n_urls = ...
n_emoji = ...
n_cyr = ...
n_dup = ...

report = pd.Series({"строк": n_rows, "пустых": n_empty, "с HTML": n_html, "со ссылками": n_urls,
                    "с эмодзи": n_emoji, "кириллица": n_cyr, "дубликатов": n_dup})
print(report)
raw.head(10)

## Задание 2. HTML: регулярное выражение или BeautifulSoup

Напишите две функции удаления HTML (таблица `raw` уже загружена):

1. `strip_html_re(text)`: `html.unescape` (превращает `&nbsp;` и другие сущности в символы), затем каждый тег `<...>` заменяется **пробелом**, затем пробельные символы схлопываются в один пробел и обрезаются по краям (удобно: `" ".join(text.split())`).
2. `strip_html_bs(text)`: `BeautifulSoup(text, "html.parser").get_text(" ")` и такое же схлопывание пробелов.

Примените обе к `raw["raw_text"].fillna("")`: результаты — Series `html_re` и `html_bs`; `n_same` — в скольких строках они совпали.

<details><summary>💡 Подсказка</summary>

Регулярка: `re.sub(r"<[^>]+>", " ", text)`. Схлопывание: `" ".join(text.split())` — `split()` без аргументов делит по любым пробелам, включая неразрывный. Применение: `texts.map(strip_html_re)`; совпадения: `(html_re == html_bs).sum()`.

</details>

In [ ]:
# Подготовительный код задания
import re, html, time
import pandas as pd
from collections import Counter

raw = pd.read_csv("data/news_raw.csv")

In [ ]:
from bs4 import BeautifulSoup

def strip_html_re(text):
    text = html.unescape(text)
    text = ...
    return ...

def strip_html_bs(text):
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    return ...

sample = "<p>Salom&nbsp;<b>dunyo</b>!</p>"
print(repr(strip_html_re(sample)), repr(strip_html_bs(sample)))

texts = raw["raw_text"].fillna("")
html_re = ...
html_bs = ...
n_same = ...
print("Совпадают:", n_same, "из", len(texts))
html_re.head(10)

## Задание 3. Функция полной очистки clean_text

Соберите функцию `clean_text(text)`. Регулярные выражения уже даны в стартовом коде, функция `normalize` — в подготовленном. Порядок шагов:

1. не строка (NaN) → вернуть `""`;
2. `html.unescape`;
3. заменить **пробелом** по очереди: теги `TAG`, служебные фразы `BOILER`, ссылки `URL`, хештеги `HASHTAG`, эмодзи `EMOJI`;
4. `normalize` (апострофы) и `.lower()`;
5. схлопнуть пробельные символы в один пробел, обрезать края.

Примените функцию ко всей таблице: `raw["clean"] = raw["raw_text"].map(clean_text)`.

<details><summary>💡 Подсказка</summary>

Цикл по шаблонам: `for rx in (TAG, BOILER, URL, HASHTAG, EMOJI): text = rx.sub(" ", text)`. Затем `text = normalize(text).lower()` и `re.sub(r"\s+", " ", text).strip()`.

</details>

In [ ]:
# Подготовительный код задания
import re, html, time
import pandas as pd
from collections import Counter

raw = pd.read_csv("data/news_raw.csv")

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

In [ ]:
TAG = re.compile(r"<[^>]+>")                                                  # HTML-теги
BOILER = re.compile(r"\(manba:[^)]*\)|Batafsil:|YANGILIK:", re.IGNORECASE)   # служебные фразы
URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)                     # ссылки
HASHTAG = re.compile(r"#\w+")                                                 # хештеги
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")                   # эмодзи

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    ...
    return text

tests = ['<p>Toshkentdagi  "Milliy"  stadionida o\u0060tgan uchrashuv. #yangilik ' + chr(0x1F44D) + "</p>",
         "<b>YANGILIK:</b> SUN\u2019IY INTELLEKT FANLARI. Batafsil: https://t.me/yangiliklar_uz",
         "&nbsp;Bu yil hosil mo\u2018l bo\u2018ladi (manba: www.talaba-news.uz)"]
for t in tests:
    print(repr(clean_text(t)))

raw["clean"] = ...
raw[["raw_text", "clean"]].head(15)

## Задание 4. Пустые строки, точные и нечёткие дубликаты

В таблицу `news` к исходным данным добавлены 20 «перепостов» (`source == "repost"`): те же новости, но с другим HTML, регистром, апострофами, эмодзи и ссылками. Функция `clean_text` уже определена.

Выполните дедупликацию строго в таком порядке и посчитайте, сколько строк удалил каждый шаг:

1. `news["clean"]` = `clean_text` от `raw_text`;
2. удалить строки с пустым `clean` → `n_empty`;
3. удалить точные дубликаты по `raw_text` (оставляя первое вхождение) → `n_dup_exact`;
4. удалить дубликаты по `clean` → `n_dup_norm`; результат — `news_dedup`;
5. `n_reposts_caught` — сколько из удалённых на шаге 4 строк — перепосты.

<details><summary>💡 Подсказка</summary>

`step1 = news[news["clean"] != ""]`; `step2 = step1.drop_duplicates(subset="raw_text")`; `news_dedup = step2.drop_duplicates(subset="clean")`. Перепосты: `(removed["source"] == "repost").sum()`.

</details>

In [ ]:
# Подготовительный код задания
import re, html, time
import pandas as pd
from collections import Counter

raw = pd.read_csv("data/news_raw.csv")

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

TAG = re.compile(r"<[^>]+>")                                                  # HTML-теги
BOILER = re.compile(r"\(manba:[^)]*\)|Batafsil:|YANGILIK:", re.IGNORECASE)   # служебные фразы
URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)                     # ссылки
HASHTAG = re.compile(r"#\w+")                                                 # хештеги
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")                   # эмодзи

def clean_text(text):
    """Полная очистка одного текста (эталон задания 3)."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    for rx in (TAG, BOILER, URL, HASHTAG, EMOJI):
        text = rx.sub(" ", text)
    text = normalize(text).lower()
    return re.sub(r"\s+", " ", text).strip()

_ok = raw["raw_text"].notna() & ~raw["raw_text"].fillna("").str.contains("&") & (raw["raw_text"].fillna("").str.len() > 30)
_src = raw[_ok].drop_duplicates("raw_text").sample(20, random_state=5)
_noise = [lambda t: "<p>" + t.upper() + "</p> " + chr(0x1F525),
          lambda t: "  " + t.replace("'", "\u2019") + "  #yangilik",
          lambda t: "<div>" + t + "</div> https://t.me/uz_news",
          lambda t: t.lower() + " " + chr(0x1F60A)]
_rep = _src.copy()
_rep["raw_text"] = [_noise[i % 4](t) for i, t in enumerate(_src["raw_text"])]
_rep["id"] = range(2001, 2001 + len(_rep))
_rep["source"] = "repost"
news = pd.concat([raw, _rep], ignore_index=True)

In [ ]:
news["clean"] = ...

step1 = news[...]
n_empty = len(news) - len(step1)

step2 = ...
n_dup_exact = ...

news_dedup = ...
n_dup_norm = ...

removed = step2[step2.duplicated(subset="clean")]
n_reposts_caught = ...

print(f"Было {len(news)}: пустых {n_empty}, точных дублей {n_dup_exact}, нечётких {n_dup_norm} -> осталось {len(news_dedup)}")
print("Найдено перепостов:", n_reposts_caught)
removed[["source", "raw_text", "clean"]].head(8)

## Задание 5. Транслитерация: кириллица → латиница

Напишите функцию `cyr2lat(text)` по официальным правилам (таблица в теории). Допишите словарь `CYR2LAT` (буквы ў, ғ, қ, ҳ, ш, ч, ё, ю, я) и обработайте особые случаи:

- **е** → `ye` в начале слова (предыдущий символ — не буква), после гласной (`VOWELS`) и после ъ/ь; иначе `e`;
- **ц** → `ts` после гласной, иначе `s`;
- заглавная кириллическая буква → первая буква латинского сочетания заглавная (Ш → Sh, Ў → Oʻ);
- остальные символы (латиница, цифры, пунктуация) не меняются.

Затем:
- `lat_texts` — словарь `{имя_файла: латинский текст}` для файлов `data/corpus_cyr/*.txt`;
- `n_cyr_rows` — число строк `raw` с кириллицей; `raw["text_lat"]` — `cyr2lat` от `raw_text` (NaN → `""`).

<details><summary>💡 Подсказка</summary>

Для е: `"ye" if (not prev.isalpha() or prev in VOWELS or prev in "ъь") else "e"`; для ц: `"ts" if prev in VOWELS else "s"`. Заглавная: `lat[0].upper() + lat[1:]`. Файлы: `{os.path.basename(p): cyr2lat(open(p, encoding="utf-8").read()) for p in sorted(glob.glob("data/corpus_cyr/*.txt"))}`.

</details>

In [ ]:
# Подготовительный код задания
import re, html, time
import pandas as pd
from collections import Counter

raw = pd.read_csv("data/news_raw.csv")

In [ ]:
import glob, os

OK, TUTUQ = "\u02bb", "\u02bc"
CYR2LAT = {
    "а": "a", "б": "b", "в": "v", "г": "g", "д": "d", "ж": "j", "з": "z", "и": "i", "й": "y",
    "к": "k", "л": "l", "м": "m", "н": "n", "о": "o", "п": "p", "р": "r", "с": "s", "т": "t",
    "у": "u", "ф": "f", "х": "x", "э": "e", "ъ": TUTUQ, "ь": "",
    # допишите: ў, ғ, қ, ҳ, ш, ч, ё, ю, я   (е и ц обрабатываются отдельно)
}
VOWELS = "аеёиоуэюяў"

def cyr2lat(text):
    out = []
    for i, ch in enumerate(text):
        low = ch.lower()
        prev = text[i - 1].lower() if i > 0 else " "
        if low == "е":
            lat = ...
        elif low == "ц":
            lat = ...
        elif low in CYR2LAT:
            lat = CYR2LAT[low]
        else:
            out.append(ch)          # не кириллица — без изменений
            continue
        if ch != low and lat:       # заглавная буква
            lat = ...
        out.append(lat)
    return "".join(out)

for w in ["Ўзбекистон", "ғалаба", "Ёшлар", "поезд", "келди", "Транслитерация", "цирк", "маъно"]:
    print(w, "->", cyr2lat(w))

lat_texts = ...
n_cyr_rows = ...
raw["text_lat"] = ...
print(lat_texts["ozbek_tili_raqamli.txt"])

## Задание 6. 100 000 строк: apply, .str и уникальные значения

В `big` — 100 000 строк (выборка с повторами из `news_raw`, как реальный поток постов, где тексты часто повторяются). Функция `clean_text` и шаблоны `TAG, BOILER, URL, HASHTAG, EMOJI` уже определены. Сравните три способа очистки столбца `big["raw_text"]` и замерьте время каждого (`time.perf_counter()`):

1. `res_apply` — `.map(clean_text)` для каждой строки; время — `t_apply`;
2. `res_vec` — функция `clean_vec(s)`, которая делает те же шаги **только методами `.str`** (`fillna`, `.str.replace(..., regex=True)`, `.str.lower()`, `.str.strip()`); сущности вида `&nbsp;` замените пробелом шаблоном `&\w+;`; время — `t_vec`;
3. `res_unique` — очистить только **уникальные** тексты (`unique()`), сложить результаты в словарь и разнести через `.map(словарь)`; время — `t_unique`.

Результаты должны совпадать. Выведите время и ускорение `t_apply / t_unique`.

<details><summary>💡 Подсказка</summary>

В цикле: `s = s.str.replace(rx, " ", regex=True)` (можно передавать скомпилированный шаблон). Апострофы: `s.str.replace(r"['\u2018\u2019\u0060]", "\u02bc", regex=True)`. Пробелы: `s.str.replace(r"\s+", " ", regex=True).str.strip()`. Словарь: `{t: clean_text(t) for t in uniq}`, затем `s.fillna("").map(mapping)`.

</details>

In [ ]:
# Подготовительный код задания
import re, html, time
import pandas as pd
from collections import Counter

raw = pd.read_csv("data/news_raw.csv")

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

TAG = re.compile(r"<[^>]+>")                                                  # HTML-теги
BOILER = re.compile(r"\(manba:[^)]*\)|Batafsil:|YANGILIK:", re.IGNORECASE)   # служебные фразы
URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)                     # ссылки
HASHTAG = re.compile(r"#\w+")                                                 # хештеги
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")                   # эмодзи

def clean_text(text):
    """Полная очистка одного текста (эталон задания 3)."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    for rx in (TAG, BOILER, URL, HASHTAG, EMOJI):
        text = rx.sub(" ", text)
    text = normalize(text).lower()
    return re.sub(r"\s+", " ", text).strip()

big = raw.sample(100_000, replace=True, random_state=42).reset_index(drop=True)

In [ ]:
s = big["raw_text"]
print("Строк:", len(s), "уникальных текстов:", s.nunique())

t0 = time.perf_counter()
res_apply = ...
t_apply = time.perf_counter() - t0

def clean_vec(s):
    s = s.fillna("")
    s = s.str.replace(r"&\w+;", " ", regex=True)
    for rx in (TAG, BOILER, URL, HASHTAG, EMOJI):
        s = ...
    s = s.str.replace(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", regex=True)
    s = ...                      # остальные апострофы -> ʼ
    s = ...                      # нижний регистр
    s = ...                      # пробелы
    return s

t0 = time.perf_counter()
res_vec = clean_vec(s)
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
uniq = s.fillna("").unique()
mapping = ...
res_unique = ...
t_unique = time.perf_counter() - t0

print(f"apply: {t_apply:.2f} c, .str: {t_vec:.2f} c, unique+map: {t_unique:.3f} c")
print("Совпадают:", (res_apply == res_vec).mean(), (res_apply == res_unique).all())
print(f"Ускорение unique+map: в {t_apply / t_unique:.0f} раз")

## Задание 7. Обработка по частям: chunksize и генератор

Представьте, что файл `news_100k.csv` (100 000 строк) не помещается в память. Обработайте его по частям.

1. Напишите **генератор** `clean_chunks(path, chunksize=20_000)`: читает файл через `pd.read_csv(path, chunksize=chunksize)`, в каждой части добавляет столбец `clean` (используйте приём «уникальные значения + map» из прошлого задания) и отдаёт часть через `yield`.
2. Пройдите по всем частям и:
   - посчитайте `n_chunks` и `n_rows`;
   - накопите частоты слов `word_freq` (`Counter`) по токенам `\w+` столбца `clean`;
   - допишите столбцы `id` и `clean` каждой части в `news_100k_clean.csv` (`mode="a"`, заголовок только у первой части, без индекса).
3. Выведите 10 самых частых слов.

<details><summary>💡 Подсказка</summary>

Генератор: `for chunk in pd.read_csv(path, chunksize=chunksize): ... yield chunk`. Частоты: `word_freq.update(words.value_counts().to_dict())`. Запись: `chunk[["id", "clean"]].to_csv("news_100k_clean.csv", mode="a", header=(i == 0), index=False)`.

</details>

In [ ]:
# Подготовительный код задания
import re, html, time
import pandas as pd
from collections import Counter

raw = pd.read_csv("data/news_raw.csv")

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

TAG = re.compile(r"<[^>]+>")                                                  # HTML-теги
BOILER = re.compile(r"\(manba:[^)]*\)|Batafsil:|YANGILIK:", re.IGNORECASE)   # служебные фразы
URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)                     # ссылки
HASHTAG = re.compile(r"#\w+")                                                 # хештеги
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")                   # эмодзи

def clean_text(text):
    """Полная очистка одного текста (эталон задания 3)."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    for rx in (TAG, BOILER, URL, HASHTAG, EMOJI):
        text = rx.sub(" ", text)
    text = normalize(text).lower()
    return re.sub(r"\s+", " ", text).strip()

import os
if not os.path.exists("news_100k.csv"):          # файл ~12 МБ, 100 000 строк
    raw.sample(100_000, replace=True, random_state=42).to_csv("news_100k.csv", index=False)
if os.path.exists("news_100k_clean.csv"):
    os.remove("news_100k_clean.csv")

In [ ]:
def clean_chunks(path, chunksize=20_000):
    for chunk in ...:
        texts = chunk["raw_text"].fillna("")
        mapping = ...
        chunk["clean"] = ...
        ...

n_chunks = 0
n_rows = 0
word_freq = Counter()
for i, chunk in enumerate(clean_chunks("news_100k.csv")):
    n_chunks += 1
    n_rows += ...
    words = chunk["clean"].str.findall(r"\w+").explode().dropna()
    word_freq.update(...)
    ...                      # дописать chunk[["id", "clean"]] в news_100k_clean.csv

print("Частей:", n_chunks, "строк:", n_rows)
print(word_freq.most_common(10))

## Самостоятельная работа

**Тема для самостоятельной работы: «Методы очистки больших объёмов данных».**

1. Сравните скорость регулярных выражений и BeautifulSoup на 10 000 HTML-страниц (сгенерируйте их или скачайте), сделайте вывод, когда что применять.
2. Реализуйте поиск нечётких дубликатов через хеширование нормализованного текста (`hashlib.md5`) и оцените, сколько памяти экономит хранение хешей вместо строк.
3. Напишите обратную транслитерацию латиница → кириллица и проверьте, что `lat2cyr(cyr2lat(text)) == text` для текстов `data/corpus_cyr`. Какие случаи неоднозначны (например, sh — это ш или с+ҳ)?